In [1]:
import ee
import xarray
import rioxarray
import rasterio
from rasterio.transform import from_origin
import threading
import warnings
import dask.dataframe as dd
import httplib2
from dask.distributed import Client
import json
import os

In [2]:
#http_transport = httplib2.Http(disable_ssl_certificate_validation=True)
#ee.Initialize(
#    opt_url='https://earthengine-highvolume.googleapis.com',
#    http_transport=http_transport, project='calfuels')
json_key = "test_key.json"
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

In [3]:
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com', project='calfuels')

In [3]:
def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)



In [4]:
initialize_dict = {
    'opt_url': 'https://earthengine-highvolume.googleapis.com',
    'project': 'calfuels'
}

In [4]:
initialize_dict = {
    'credentials': credentials,
    'opt_url': 'https://earthengine-highvolume.googleapis.com'
}

In [ ]:
client = Client(n_workers=2, memory_limit="8GiB", threads_per_worker=1)
client.run(ee.Initialize, **initialize_dict)

2024-06-12 13:00:45,546 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 0 memory: 268 MB fds: N/A>>
Traceback (most recent call last):
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\compatibility.py", line 161, in _run
    val = self.callback()
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\system_monitor.py", line 169, in update
    gpu_metrics = nvml.real_time()
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\diagnostics\nvml.py", line 327, in real_time
    "utilization": _get_utilization(h),
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\diagnostics\nvml.py", line 298, in _get_utilization
    return pynvml.nvmlDeviceGetUtilizationRates(h).gpu
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\pynvml\nvml.py", line 2137, in nvmlDeviceGetUtilizationRates
    _nvmlCheckReturn(ret)


{'tcp://127.0.0.1:57039': None, 'tcp://127.0.0.1:57040': None}

2024-06-12 13:00:52,039 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 9 memory: 270 MB fds: N/A>>
Traceback (most recent call last):
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\compatibility.py", line 161, in _run
    val = self.callback()
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\system_monitor.py", line 169, in update
    gpu_metrics = nvml.real_time()
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\diagnostics\nvml.py", line 327, in real_time
    "utilization": _get_utilization(h),
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\diagnostics\nvml.py", line 298, in _get_utilization
    return pynvml.nvmlDeviceGetUtilizationRates(h).gpu
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\pynvml\nvml.py", line 2137, in nvmlDeviceGetUtilizationRates
    _nvmlCheckReturn(ret)


2024-06-12 13:00:52,444 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 0 memory: 269 MB fds: N/A>>
Traceback (most recent call last):
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\compatibility.py", line 161, in _run
    val = self.callback()
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\system_monitor.py", line 169, in update
    gpu_metrics = nvml.real_time()
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\diagnostics\nvml.py", line 327, in real_time
    "utilization": _get_utilization(h),
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\distributed\diagnostics\nvml.py", line 298, in _get_utilization
    return pynvml.nvmlDeviceGetUtilizationRates(h).gpu
  File "c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\pynvml\nvml.py", line 2137, in nvmlDeviceGetUtilizationRates
    _nvmlCheckReturn(ret)


In [6]:
client = Client(n_workers=2, memory_limit="16GiB", threads_per_worker=2)
client.run(ee.Initialize, **initialize_dict)

{'tcp://127.0.0.1:64321': None, 'tcp://127.0.0.1:64324': None}

In [20]:
client.close()

In [4]:
PREFERRED_CHUNKS = {
    'time': 48,
    'X': 512,
    'Y': 256
}

In [6]:
FAILING_CHUNKS = {
    'time': 1024,
    'X': 1024,
    'Y': 1024
}

In [6]:
CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterDate('2000-01-01', '2022-12-31').filterBounds(LTBMU.geometry())
first_img = ic.first()
ic_first = ee.ImageCollection(first_img)
ic_xr = xarray.open_dataset(ic, engine = "ee", chunks=PREFERRED_CHUNKS, crs='EPSG:3310', geometry=LTBMU.geometry(), scale = 100)

In [7]:
CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

dem_img = ee.Image("USGS/3DEP/10m")
ic_first = ee.ImageCollection(dem_img)
ic_xr = xarray.open_dataset(ic_first, engine = "ee", crs='EPSG:3310', geometry=LTBMU.geometry(), scale = 10)

c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\xee\ext.py:683: UserWarning: Unable to retrieve 'system:time_start' values from an ImageCollection due to: No 'system:time_start' values found in the 'ImageCollection'.
  warnings.warn(


In [8]:
print(ic_xr)

<xarray.Dataset>
Dimensions:    (time: 1, X: 3252, Y: 6533)
Coordinates:
  * time       (time) int32 0
  * X          (X) float64 -2.191e+04 -2.19e+04 ... 1.059e+04 1.06e+04
  * Y          (Y) float64 7.649e+04 7.65e+04 7.651e+04 ... 1.418e+05 1.418e+05
Data variables:
    elevation  (time, X, Y) float32 ...
Attributes:
    crs:      EPSG:3310


In [18]:
ndvi = (ic_xr['SR_B5'] - ic_xr['SR_B4']) / (ic_xr['SR_B5'] + ic_xr['SR_B4'])
ic_xr['NDVI'] = ndvi

In [8]:
print(ic_xr["SR_B1"])

<xarray.DataArray 'SR_B1' (time: 218, X: 325, Y: 653)>
dask.array<open_dataset-fef8215d5980696a802aa94f832bf0dcSR_B1, shape=(218, 325, 653), dtype=float32, chunksize=(48, 325, 256), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 2013-03-22T18:36:56.975000 ... 2013-04-09T...
  * X        (X) float64 -2.187e+04 -2.177e+04 ... 1.043e+04 1.053e+04
  * Y        (Y) float64 7.654e+04 7.664e+04 7.674e+04 ... 1.416e+05 1.417e+05
Attributes:
    id:             SR_B1
    data_type:      {'type': 'PixelType', 'precision': 'double', 'min': -0.2,...
    dimensions:     [7591, 7541]
    crs:            EPSG:3310
    crs_transform:  [30, 0, 192585, 0, -30, 4421115]


In [19]:
ic_xr_result = ic_xr.compute()

KeyboardInterrupt: 

In [51]:
print(ic_xr_result)

<xarray.Dataset>
Dimensions:        (time: 22, X: 1084, Y: 2178)
Coordinates:
  * time           (time) datetime64[ns] 2019-01-04T18:39:22.062000 ... 2019-...
  * X              (X) float64 -2.19e+04 -2.187e+04 ... 1.056e+04 1.059e+04
  * Y              (Y) float64 7.65e+04 7.653e+04 ... 1.418e+05 1.418e+05
Data variables: (12/20)
    SR_B1          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B2          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B3          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B4          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B5          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B6          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    ...             ...
    ST_QA          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    ST_TRAD        (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
   

In [52]:
ic_xr_result = ic_xr_result.rename({'X': 'x', 'Y': 'y'})
ic_xr_result = ic_xr_result.transpose('time', 'y', 'x')

In [54]:
%%time
ic_xr_result["NDVI"].isel(time=0).rio.to_raster("ndvi.tif")

CPU times: total: 62.5 ms
Wall time: 48.8 ms


In [ ]:
df = ic_xr.to_dask_dataframe()

In [ ]:
df.map_partitions(len).compute()

In [ ]:
print(df)

In [8]:
# Define a function to calculate NDVI
def calculate_ndvi(row):
    red = row['SR_B4']
    nir = row['SR_B5']
    ndvi = (nir - red) / (nir + red)
    return ndvi

In [14]:
# Define a function to calculate NDVI
def calculate_ndvi(red_col, nir_col):
    ndvi = (nir_col - red_col) / (nir_col + red_col)
    return ndvi

In [10]:
df['NDVI'] = df.apply(calculate_ndvi, axis=1, meta=('ndvi', 'f8'))

In [15]:
df['NDVI'] = dd.map_partitions(calculate_ndvi, df['SR_B4'], df['SR_B5'])

In [19]:
df = df.compute()

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
# Lazy operation! Does not load the raster into memory!!
da = xarray.open_dataset("https://github.com/mapbox/rasterio/raw/1.2.1/tests/data/RGB.byte.tif")

In [ ]:
print(da)

In [12]:
print(ic_xr.coarsen.__doc__)


        Coarsen object for Datasets.

        Parameters
        ----------
        dim : mapping of hashable to int, optional
            Mapping from the dimension name to the window size.
        boundary : {"exact", "trim", "pad"}, default: "exact"
            If 'exact', a ValueError will be raised if dimension size is not a
            multiple of the window size. If 'trim', the excess entries are
            dropped. If 'pad', NA will be padded.
        side : {"left", "right"} or mapping of str to {"left", "right"}, default: "left"
        coord_func : str or mapping of hashable to str, default: "mean"
            function (name) that is applied to the coordinates,
            or a mapping from coordinate name to function (name).

        Returns
        -------
        core.rolling.DatasetCoarsen

        See Also
        --------
        core.rolling.DatasetCoarsen
        DataArray.coarsen

        :ref:`reshape.coarsen`
            User guide describing :py:func:`~xarray.D

In [ ]:
factor = 100 // 30
resample = ic_xr.coarsen(X=factor, Y=factor).mean()

In [ ]:
warnings.filterwarnings("ignore")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU")

#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'], 
          #['UBlue', 'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']).filterBounds(LTBMU.geometry())

ic_landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterBounds(LTBMU.geometry())
ic_dem = ee.ImageCollection("USGS/3DEP/1m").filterBounds(LTBMU.geometry())
ic_xr = xarray.open_mfdataset([ic_landsat, ic_dem], engine = "ee", crs='EPSG:3310', scale = 30)



In [ ]:
ds = xarray.open_mfdataset(['ee://ECMWF/ERA5_LAND/HOURLY', 'ee://NASA/GDDP-CMIP6'],
                           engine='ee', crs='EPSG:4326', scale=0.25)

In [5]:
red_band = ic_xr['SR_B4'].isel(time=0).chunk(chunks={"X": 1000, "Y": 1000})
nir_band = ic_xr['SR_B5'].isel(time=0).chunk(chunks={"X": 1000, "Y": 1000})
ndvi = (nir_band - red_band) / (nir_band + red_band)
ndvi_transpose = ndvi.transpose('Y', 'X').rename({"X": "x", "Y": "y"})

In [ ]:
print(ndvi_transpose.rio.to_raster.__doc__)

In [ ]:
print(ic_xr['SR_B5'].isel(time=0).chunk.__doc__)

In [ ]:
print(ndvi_transpose.to_netcdf.__doc__)

In [6]:
ndvi_transpose.rio.to_raster(raster_path="ndvi.tif", windowed=True, tiled=True, lock=threading.Lock())

c:\Users\knigh\anaconda3\envs\ee\Lib\site-packages\dask\core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


In [30]:
ic = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate('1992-10-05', '1993-03-31')
leg1 = ee.Geometry.Rectangle(113.33, -43.63, 153.56, -10.66)
ds = xarray.open_dataset(
    ic,
    engine='ee',
    projection=ic.first().select(0).projection(),
    geometry=leg1
)

In [ ]:
ds_renamed = ds.rename({'lon': 'x', 'lat': 'y'})
ds_renamed_transpose = ds_renamed.transpose('time', 'y', 'x')
ds_renamed_transpose.isel(time=0).rio.to_raster("hourly.tif")
#ds_renamed_transpose.rio.to_raster(output_raster_path, driver='GTiff', crs='EPSG:4326', compress='lzw')

In [ ]:
'''ds_img = ds.sel(time='1992-10-05T00:00:00.000000000')
print(ds_img.coords)
lat_range = slice(-10, -20.0)  # Replace with your desired latitude range
lon_range = slice(120, 130)  # Replace with your desired longitude range

# Use .sel to filter the dataset
tiled_img = ds_img.sel(lat=lat_range, lon=lon_range)
print(tiled_img.coords)
# Print the filtered dataset
#print(tiled_img)'''

In [ ]:
def calculate_ndvi(red, nir):
    return (nir - red) / (nir + red)

# Apply the custom function to compute NDVI
ndvi = xarray.apply_ufunc(
    calculate_ndvi,
    ic_xr['SR_B4'],  # Red band variable
    ic_xr['SR_B5'],  # Near-Infrared band variable
    dask='parallelized',  # Use dask if you want to handle large datasets
    input_core_dims=[[], []],  # The dimensions of the input variables
    output_core_dims=[[]],  # The dimension of the output variable
    vectorize=True,  # Whether to vectorize the function
    output_dtypes=[float],  # The dtype of the output variable
)

ic_xr['NDVI'] = ndvi.compute()